
# Exact Inference in Gaussian Models

In the discrete case, exact inference was performed by manipulating factors:

```text
Factors
   ↓
Multiply
   ↓
Eliminate hidden variables
   ↓
Obtain query distribution
```

The same high-level idea applies to Gaussian models.

The difference is that instead of probability tables, Gaussian factors are represented using quadratic functions or canonical parameters.

This notebook develops the Gaussian analogue of Variable Elimination.

We will study:

- Gaussian factors,
- canonical-form factor representation,
- multiplication of Gaussian factors,
- elimination of variables,
- exact inference in a small Gaussian graphical model,
- the connection to discrete Variable Elimination.

The main goal is to show that **the inference algorithm is conceptually the same; only the algebra changes**.



## 1. Inference Flowchart

```text
Gaussian graphical model
        │
        ▼
Gaussian factors
        │
        ▼
Convert / represent factors
in canonical form
        │
        ▼
Multiply relevant factors
        │
        ▼
Eliminate hidden variables
        │
        ▼
Remaining Gaussian factor
        │
        ▼
Normalize / convert to moment form
        │
        ▼
Query distribution
```

Compare this with discrete Variable Elimination:

```text
Discrete factors
      ↓
Multiply
      ↓
Sum out hidden variables
      ↓
Normalize
```

The structure is nearly identical.



## 2. Why Canonical Form Is Useful Here

A Gaussian in canonical form is

$$
p(\mathbf{x})
\propto
\exp
\left(
-\frac{1}{2}
\mathbf{x}^{\top}\Lambda\mathbf{x}
+
\boldsymbol{\eta}^{\top}\mathbf{x}
\right).
$$

For two Gaussian factors,

$$
(\eta_1,\Lambda_1)
$$

and

$$
(\eta_2,\Lambda_2),
$$

multiplication gives

$$
\boxed{
\Lambda=
\Lambda_1+\Lambda_2
}
$$

and

$$
\boxed{
\eta=
\eta_1+\eta_2.
}
$$

So factor multiplication becomes addition of information.

This is one of the main reasons information form is attractive for Gaussian graphical models.


In [1]:

import numpy as np

np.set_printoptions(
    precision=4,
    suppress=True,
)



# 3. Gaussian Factors

A factor is a function over a subset of variables.

For example,

$$
\phi(X,Y)
$$

depends on $X$ and $Y$, while

$$
\psi(Y,Z)
$$

depends on $Y$ and $Z$.

In the discrete case, a factor stores a table.

In the Gaussian case, a canonical Gaussian factor can store:

- its variable scope,
- an information vector $\eta$,
- a precision matrix $\Lambda$.

The order of variables determines the order of entries in $\eta$ and the rows/columns of $\Lambda$.



## 3.1 A Minimal GaussianFactor Class

We will first define a small factor class that stores canonical parameters.

For simplicity, this notebook treats each variable as scalar.

Later, the same idea can be generalized to vector-valued variables.


In [3]:

from dataclasses import dataclass


@dataclass
class GaussianFactor:
    variables: list[str]
    information: np.ndarray
    precision: np.ndarray

    def __post_init__(self) -> None:
        self.variables = list(self.variables)

        self.information = np.asarray(
            self.information,
            dtype=float,
        )

        self.precision = np.asarray(
            self.precision,
            dtype=float,
        )

        self._validate_parameters()

    @property
    def dimension(self) -> int:
        return len(self.variables)

    def _validate_parameters(self) -> None:
        if self.dimension == 0:
            raise ValueError(
                "A factor must contain at least one variable."
            )

        if len(set(self.variables)) != self.dimension:
            raise ValueError(
                "Variable names must be unique."
            )

        if self.information.shape != (
            self.dimension,
        ):
            raise ValueError(
                "Information vector has incompatible shape."
            )

        if self.precision.shape != (
            self.dimension,
            self.dimension,
        ):
            raise ValueError(
                "Precision matrix has incompatible shape."
            )

        if not np.allclose(
            self.precision,
            self.precision.T,
        ):
            raise ValueError(
                "Precision matrix must be symmetric."
            )



# 4. Factor Multiplication

Suppose

$$
\phi_1(X,Y)
$$

and

$$
\phi_2(Y,Z)
$$

share variable \(Y\).

Before adding their canonical parameters, both factors must be embedded into the same variable ordering.

For example, choose

$$
[X,Y,Z].
$$

Then:

- $\phi_1$ contributes information to $X,Y$,
- $\phi_2$ contributes information to $Y,Z$,
- missing entries are filled with zeros.

Once aligned,

$$
\Lambda=
\Lambda_1+\Lambda_2,
$$

$$
\eta=
\eta_1+\eta_2.
$$



## 4.1 Aligning a Factor to a Larger Scope


In [4]:

def align_factor(
    factor: GaussianFactor,
    target_variables: list[str],
) -> tuple[np.ndarray, np.ndarray]:
    target_variables = list(
        target_variables
    )

    target_dimension = len(
        target_variables
    )

    information = np.zeros(
        target_dimension
    )

    precision = np.zeros(
        (
            target_dimension,
            target_dimension,
        )
    )

    index_map = {
        variable: index
        for index, variable
        in enumerate(target_variables)
    }

    factor_indices = [
        index_map[variable]
        for variable in factor.variables
    ]

    information[
        factor_indices
    ] = factor.information

    precision[
        np.ix_(
            factor_indices,
            factor_indices,
        )
    ] = factor.precision

    return information, precision



## 4.2 Multiplication Implementation


In [6]:

def multiply_factors(
    factor_1: GaussianFactor,
    factor_2: GaussianFactor,
) -> GaussianFactor:
    variables = list(
        dict.fromkeys(
            factor_1.variables
            + factor_2.variables
        )
    )

    information_1, precision_1 = (
        align_factor(
            factor_1,
            variables,
        )
    )

    information_2, precision_2 = (
        align_factor(
            factor_2,
            variables,
        )
    )

    combined_information = (
        information_1
        + information_2
    )

    combined_precision = (
        precision_1
        + precision_2
    )

    return GaussianFactor(
        variables=variables,
        information=combined_information,
        precision=combined_precision,
    )



# 5. Gaussian Variable Elimination

Now comes the key step.

Suppose a canonical Gaussian factor is partitioned as

$$
\mathbf{x}=
\begin{bmatrix}
\mathbf{x}_a\\
\mathbf{x}_b
\end{bmatrix}
$$

where:

- $\mathbf{x}_a$ are variables we want to retain,
- $\mathbf{x}_b$ are variables we want to eliminate.

Partition the canonical parameters:

$$
\Lambda=
\begin{bmatrix}
\Lambda_{aa} & \Lambda_{ab}\\
\Lambda_{ba} & \Lambda_{bb}
\end{bmatrix}
$$

and

$$
\eta=
\begin{bmatrix}
\eta_a\\
\eta_b
\end{bmatrix}.
$$

After integrating out $\mathbf{x}_b$, the resulting canonical parameters are

$$
\boxed{
\Lambda'=
\Lambda_{aa}-
\Lambda_{ab}
\Lambda_{bb}^{-1}
\Lambda_{ba}
}
$$

and

$$
\boxed{
\eta'=
\eta_a-
\Lambda_{ab}
\Lambda_{bb}^{-1}
\eta_b
}
$$

This is the Gaussian equivalent of summing out a variable in a discrete factor.



## 5.1 Intuition

The structure

$$
\Lambda_{aa}-
\Lambda_{ab}
\Lambda_{bb}^{-1}
\Lambda_{ba}
$$

is a **Schur complement**.

The eliminated variable carries some information that couples to the retained variables.

Integrating it out removes that variable explicitly, but its influence survives as a correction to the remaining precision and information vector.

So elimination does **not** mean "delete the corresponding rows and columns."

That simple slicing rule applied to marginalization in **moment form**.

In canonical form, marginalization requires the Schur-complement correction.



## 5.2 Elimination Implementation


In [7]:

def eliminate_variable(
    factor: GaussianFactor,
    variable: str,
) -> GaussianFactor:
    if variable not in factor.variables:
        raise ValueError(
            f"{variable!r} is not in the factor."
        )

    eliminate_index = (
        factor.variables.index(
            variable
        )
    )

    keep_indices = [
        index
        for index in range(
            factor.dimension
        )
        if index != eliminate_index
    ]

    if len(keep_indices) == 0:
        raise ValueError(
            "Cannot eliminate the only variable "
            "in this simplified implementation."
        )

    Lambda_aa = factor.precision[
        np.ix_(
            keep_indices,
            keep_indices,
        )
    ]

    Lambda_ab = factor.precision[
        np.ix_(
            keep_indices,
            [eliminate_index],
        )
    ]

    Lambda_ba = factor.precision[
        np.ix_(
            [eliminate_index],
            keep_indices,
        )
    ]

    Lambda_bb = factor.precision[
        np.ix_(
            [eliminate_index],
            [eliminate_index],
        )
    ]

    eta_a = factor.information[
        keep_indices
    ]

    eta_b = factor.information[
        [eliminate_index]
    ]

    reduced_precision = (
        Lambda_aa
        - Lambda_ab
        @ np.linalg.solve(
            Lambda_bb,
            Lambda_ba,
        )
    )

    reduced_information = (
        eta_a
        - Lambda_ab
        @ np.linalg.solve(
            Lambda_bb,
            eta_b,
        )
    )

    reduced_variables = [
        factor.variables[index]
        for index in keep_indices
    ]

    return GaussianFactor(
        variables=reduced_variables,
        information=reduced_information,
        precision=reduced_precision,
    )



# 6. A Small Gaussian Graphical Model

Consider a simple chain:

```text
X  ───  Y  ───  Z
```

with Gaussian factors

$$
\phi_1(X,Y)
$$

and

$$
\phi_2(Y,Z).
$$

Suppose we want the marginal over

$$
(X,Z).
$$

The hidden variable is

$$
Y.
$$

Gaussian Variable Elimination proceeds exactly like discrete VE:

```text
φ₁(X,Y)     φ₂(Y,Z)
      \       /
       \     /
        Multiply
           │
           ▼
      φ(X,Y,Z)
           │
           ▼
       Eliminate Y
           │
           ▼
        φ(X,Z)
```



## 6.1 Construct Example Factors

We choose canonical parameters that produce a valid positive-definite combined system.


In [8]:

factor_xy = GaussianFactor(
    variables=["X", "Y"],
    information=np.array([
        1.0,
        0.5,
    ]),
    precision=np.array([
        [2.0, -0.8],
        [-0.8, 1.5],
    ]),
)

factor_yz = GaussianFactor(
    variables=["Y", "Z"],
    information=np.array([
        0.2,
        1.2,
    ]),
    precision=np.array([
        [1.7, -0.6],
        [-0.6, 1.8],
    ]),
)



## 6.2 Multiply the Factors


In [9]:

joint_factor = multiply_factors(
    factor_xy,
    factor_yz,
)

print("Variables:")
print(joint_factor.variables)

print("\nInformation:")
print(joint_factor.information)

print("\nPrecision:")
print(joint_factor.precision)


Variables:
['X', 'Y', 'Z']

Information:
[1.  0.7 1.2]

Precision:
[[ 2.  -0.8  0. ]
 [-0.8  3.2 -0.6]
 [ 0.  -0.6  1.8]]



## 6.3 Eliminate the Hidden Variable $Y$


In [12]:

factor_xz = eliminate_variable(
    joint_factor,
    variable="Y",
)

print("Variables:")
print(factor_xz.variables)

print("\nInformation:")
print(factor_xz.information)

print("\nPrecision:")
print(factor_xz.precision)


Variables:
['X', 'Z']

Information:
[1.175  1.3313]

Precision:
[[ 1.8    -0.15  ]
 [-0.15    1.6875]]



# 7. Convert the Result to Moment Form

Once the final canonical factor contains only the query variables, we can convert it to mean and covariance form.

For a normalized Gaussian,

$$
\Sigma=
\Lambda^{-1}
$$

and

$$
\mu=
\Sigma\eta.
$$


In [13]:

query_covariance = np.linalg.inv(
    factor_xz.precision
)

query_mean = (
    query_covariance
    @ factor_xz.information
)

print("Query mean:")
print(query_mean)

print("\nQuery covariance:")
print(query_covariance)


Query mean:
[0.7239 0.8532]

Query covariance:
[[0.5597 0.0498]
 [0.0498 0.597 ]]



# 8. Numerical Verification

We can verify the elimination result by first converting the full joint canonical Gaussian into moment form.

Then, because marginalization in moment form is easy, we simply select the \(X\) and \(Z\) entries.

The result should match Gaussian Variable Elimination.


In [14]:

joint_covariance = np.linalg.inv(
    joint_factor.precision
)

joint_mean = (
    joint_covariance
    @ joint_factor.information
)

print("Joint mean:")
print(joint_mean)

print("\nJoint covariance:")
print(joint_covariance)


Joint mean:
[0.7239 0.5597 0.8532]

Joint covariance:
[[0.5597 0.1493 0.0498]
 [0.1493 0.3731 0.1244]
 [0.0498 0.1244 0.597 ]]


In [15]:

xz_indices = [
    joint_factor.variables.index("X"),
    joint_factor.variables.index("Z"),
]

direct_mean = joint_mean[
    xz_indices
]

direct_covariance = joint_covariance[
    np.ix_(
        xz_indices,
        xz_indices,
    )
]

print("Direct marginal mean:")
print(direct_mean)

print("\nVE marginal mean:")
print(query_mean)

print("\nDirect marginal covariance:")
print(direct_covariance)

print("\nVE marginal covariance:")
print(query_covariance)


Direct marginal mean:
[0.7239 0.8532]

VE marginal mean:
[0.7239 0.8532]

Direct marginal covariance:
[[0.5597 0.0498]
 [0.0498 0.597 ]]

VE marginal covariance:
[[0.5597 0.0498]
 [0.0498 0.597 ]]



The two results should agree up to numerical precision.

This confirms that canonical-form elimination produces the same marginal as direct Gaussian marginalization.



# 9. General Gaussian Variable Elimination

We can now describe the full algorithm.

Given:

- a collection of Gaussian factors,
- query variables,
- an elimination order,

repeat:

```text
Choose hidden variable H
        │
        ▼
Find all factors containing H
        │
        ▼
Multiply those factors
        │
        ▼
Eliminate H using Schur complement
        │
        ▼
Insert reduced factor
        │
        ▼
Continue with next hidden variable
```

At the end, multiply the remaining factors and convert the result to moment form if desired.

This is structurally the same as discrete Variable Elimination.



## 9.1 Algorithm Implementation


In [17]:

def gaussian_variable_elimination(
    factors: list[GaussianFactor],
    query_variables: list[str],
    elimination_order: list[str],
) -> GaussianFactor:
    working_factors = list(factors)

    for variable in elimination_order:
        related_factors = [
            factor
            for factor in working_factors
            if variable in factor.variables
        ]

        if len(related_factors) == 0:
            continue

        working_factors = [
            factor
            for factor in working_factors
            if variable not in factor.variables
        ]

        combined = related_factors[0]

        for factor in related_factors[1:]:
            combined = multiply_factors(
                combined,
                factor,
            )

        reduced = eliminate_variable(
            combined,
            variable,
        )

        working_factors.append(
            reduced
        )

    result = working_factors[0]

    for factor in working_factors[1:]:
        result = multiply_factors(
            result,
            factor,
        )

    if set(result.variables) != set(
        query_variables
    ):
        raise ValueError(
            "Final factor does not match query variables. "
            "Check elimination_order."
        )

    return result



## 9.2 Test the General Algorithm


In [18]:

result = gaussian_variable_elimination(
    factors=[
        factor_xy,
        factor_yz,
    ],
    query_variables=[
        "X",
        "Z",
    ],
    elimination_order=[
        "Y",
    ],
)

result_covariance = np.linalg.inv(
    result.precision
)

result_mean = (
    result_covariance
    @ result.information
)

print("Result variables:")
print(result.variables)

print("\nResult mean:")
print(result_mean)

print("\nResult covariance:")
print(result_covariance)


Result variables:
['X', 'Z']

Result mean:
[0.7239 0.8532]

Result covariance:
[[0.5597 0.0498]
 [0.0498 0.597 ]]



# 10. Discrete vs Gaussian Variable Elimination

| Step | Discrete VE | Gaussian VE |
|---|---|---|
| Factor representation | Probability table | Canonical parameters $(\eta,\Lambda)$ |
| Combine factors | Multiply table entries | Add canonical information |
| Eliminate variable | Sum over states | Integrate analytically |
| Elimination algebra | Summation | Schur complement |
| Final result | Discrete factor | Gaussian factor |
| Normalize / interpret | Normalize probabilities | Convert to moment form if needed |
| Complexity driver | Factor scope size | Matrix / clique size |
| Elimination order matters? | Yes | Yes |

The key conceptual lesson is:

> Variable Elimination is not inherently discrete.

It is a general inference strategy. The mathematical operation used to eliminate a variable depends on the representation of the factors.



# 11. Important Insight: Moment vs Canonical Form

A subtle but important distinction now becomes clear.

### Moment form

Marginalization is easy:

$$
(\mu,\Sigma)
\rightarrow
\text{select the desired blocks}.
$$

### Canonical form

Multiplication is easy:

$$
\eta=\eta_1+\eta_2,
\qquad
\Lambda=\Lambda_1+\Lambda_2.
$$

But marginalization requires a Schur complement.

This explains why both representations are useful.

```text
Moment form
    └── easy marginal interpretation

Canonical form
    └── easy factor multiplication
        and graphical-model inference
```



# 12. Final Flowchart

```text
Gaussian factors
      │
      ▼
Canonical representation
      │
      ▼
Pick hidden variable
      │
      ▼
Collect all factors
containing that variable
      │
      ▼
Multiply factors
(add η and Λ)
      │
      ▼
Eliminate variable
(Schur complement)
      │
      ▼
Create reduced factor
      │
      ▼
Repeat
      │
      ▼
Final query factor
      │
      ▼
Convert to (μ, Σ)
```



# 13. Summary

In this notebook we extended Variable Elimination from discrete models to Gaussian models.

The high-level algorithm remained unchanged:

1. identify factors involving a hidden variable,
2. combine them,
3. eliminate that variable,
4. repeat until only query variables remain.

The main differences are algebraic:

- canonical Gaussian factors replace probability tables,
- multiplication becomes addition of information parameters,
- integration is performed analytically using the Schur complement.

This provides an exact inference algorithm for Gaussian graphical models.

## Next Step

The next notebook is **Gaussian Belief Propagation**.

Instead of centrally eliminating variables one by one, we will express the same Gaussian inference ideas as local messages passed between nodes and factors.
